# Evaluation and validation

Notebook 02 showed the score is internally stable. It did not show it is *useful* — a different
claim, and the only one this notebook makes.

The score was built with hindsight: features chosen, weights set and the ranking inspected while
the detentions were already visible. V001 and V004 topping it is therefore not evidence, just the
expected result of tuning against a known answer.

The question here instead:

> **Standing at a date in the past, with everything after it hidden, would this system have 
> pointed at the vessels that later went wrong?**

In [1]:
import sys
from pathlib import Path

for _dir in (Path.cwd(), *Path.cwd().parents):
    if (_dir / 'pyproject.toml').exists():
        sys.path.insert(0, str(_dir))
        break

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from feature_builder import (
    build_features, inspection_features, defs_dated, inspections, detentions,
    VESSEL_IDS, REFERENCE_DATE,
)
from scoring import score_on, minmax, score_fleet, DIMENSIONS

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

print(f'{len(VESSEL_IDS)} vessels, {len(inspections)} inspections, {len(detentions)} detentions')

20 vessels, 80 inspections, 2 detentions


## 1. What is being measured

2 measures, in order of the weight they deserve:

| Measure | Question it answers | Weight given |
|---|---|---|
| **Recall at top-k** | **Did we catch the bad ones?**  | primary |
| **Lift at top-k** | **Was the list better than guessing?** | secondary |

Recall is prioritised over precision because of the costs: a false alert costs a
superintendent's time, a missed vessel costs the whole charter.

## 2. The back-test

Rewind to **31 December 2024**. Build the score using only inspections on or before that date,
then compare it against what happened at the inspections that followed.

A constraint have to be stated before the result, because they limit what it proves:

- **Only the inspection half can be rewound.** Maintenance due dates are all in 2026, equipment
  failures begin in March 2025, audit findings carry no dates at all. Rewound to 2024 those tables
  are empty. So the back-test scores four dimensions of the seven, and the operational half of the
  model is **not validated here**.

In [6]:
CUTOFF = '2024-12-31'

early = inspection_features(CUTOFF)

# score_on keeps only the dimensions whose features still exist at the cutoff, so which
# dimensions survive the rewind is derived rather than hand-listed
score_2024, dims_used = score_on(early)

print(f'as of {CUTOFF}: {early.n_inspections.unique().tolist()} inspections per vessel, '
      f'{early.n_deficiencies.sum()} deficiencies on record')
print('\ndimensions that survive the rewind:')
for k, v in dims_used.items():
      print(f"{k} -> {list(v['features'])}")
print('\ndropped, bcoz  no rewindable data:', [k for k in DIMENSIONS if k not in dims_used])

as of 2024-12-31: [2, 3] inspections per vessel, 163 deficiencies on record

dimensions that survive the rewind:
Repetition -> ['repeat_count', 'max_repeat']
Trend -> ['trend_clipped']
Unresolved -> ['open_deficiencies']
Concentration -> ['concentration']

dropped, bcoz  no rewindable data: ['Maintenance', 'Crew', 'Equipment']


In [12]:
# IMP: Everything here uses > CUTOFF. Everything in the previous cell used <= CUTOFF.

#Keep only deficiencies from inspections after the cutoff, then count them per vessel.
future_defs = (defs_dated[defs_dated.inspection_date > CUTOFF]
               .groupby('vessel_id').size().reindex(VESSEL_IDS).fillna(0).astype(int))
future_insp = inspections[inspections.inspection_date > CUTOFF]

# Which vessels were detained after the cutoff.
detained_later = set(future_insp[future_insp.detention_flag == 'Yes'].vessel_id)

bt = pd.DataFrame({'score_2024_predicted': score_2024})
bt['future_deficiencies_actual'] = future_defs
bt['detained_in_future_actual'] = [v in detained_later for v in bt.index]
bt = bt.sort_values('score_2024_predicted', ascending=False)
bt.insert(1, 'rank_predicted', range(1, len(bt) + 1))


print(bt.head(10).to_string())

           score_2024_predicted  rank_predicted  future_deficiencies_actual  detained_in_future_actual
vessel_id                                                                                             
V018                       64.2               1                           1                      False
V001                       58.5               2                          19                       True
V004                       50.7               3                          12                       True
V012                       46.5               4                           4                      False
V008                       41.1               5                           4                      False
V013                       40.2               6                           5                      False
V011                       34.8               7                           5                      False
V005                       34.4               8                          

### What the table says

| Vessel | Predicted | What happened | Verdict |
|---|---|---|---|
| V001 | rank 2 of 20 | 19 deficiencies, **detained** | caught |
| V004 | rank 3 of 20 | 12 deficiencies, **detained** | caught |
| V018 | rank 1 of 20 | 1 deficiency, no detention | false alarm |
| V005 | rank 8 of 20 | 1 deficiency, no detention | correctly ignored |

Both vessels that went on to be detained were placed in the top three, scored on information
available eighteen months before either detention occurred. Neither detention existed in the data
the score was built from.

The vessel ranked first was a false alarm — V018 had a backlog in December 2024 that was
subsequently closed, and almost nothing went wrong afterwards. One of the top three was wasted
attention, and that is an acceptable price.

In [28]:
# 123 — every deficiency recorded across the fleet after the cutoff.
total_actual = bt.future_deficiencies_actual.sum()
print("Testing: did the vessels we put at the top turn out to have more trouble than the ones we put at the bottom?")

print(f'\n\n{total_actual} deficiencies were recorded after {CUTOFF}, '
      f'and {len(detained_later)} vessels were detained.\n')

rows = []

for k in (3, 5, 9, 10, 11, 12, 13):
    top = bt.head(k)
    captured = top.future_deficiencies_actual.sum()
    expected = total_actual * k / len(bt)
    rows.append({
        'vessels_flagged': k,
        'deficiencies_that_followed': captured,
        'expected_at_random': round(expected, 1),
        'lift_vs_random': round(captured / expected, 2),
        'detentions_caught': f'{top.detained_in_future_actual.sum()} of {len(detained_later)}',
    })
print(pd.DataFrame(rows).to_string(index=False))

Testing: did the vessels we put at the top turn out to have more trouble than the ones we put at the bottom?


123 deficiencies were recorded after 2024-12-31, and 2 vessels were detained.

 vessels_flagged  deficiencies_that_followed  expected_at_random  lift_vs_random detentions_caught
               3                          32                18.4            1.73            2 of 2
               5                          40                30.8            1.30            2 of 2
               9                          56                55.4            1.01            2 of 2
              10                          60                61.5            0.98            2 of 2
              11                          65                67.6            0.96            2 of 2
              12                          72                73.8            0.98            2 of 2
              13                          80                80.0            1.00            2 of 2
